# 11 · Benchmark tabular pareado y Tabla suplementaria S12

Este notebook ejecuta de principio a fin el benchmark predictivo pareado añadido tras la evaluación externa de InterGATE. En una única ejecución:

1. comprueba matrices, metadatos, artefactos y predicciones externas;
2. ajusta y congela elastic net, random forest y XGBoost en el mismo `train_idx`;
3. calcula las predicciones de validación en el mismo `val_idx`;
4. aplica los modelos congelados a las mismas muestras externas, sin reajuste;
5. genera las métricas, matrices de confusión, intercambio LumA↔LumB, IC del 95 % y las tablas definitivas de S12.

La evaluación externa de InterGATE ya existe. Por ello, el manifiesto etiqueta este análisis como `posthoc_fixed_protocol`. No se seleccionan hiperparámetros con el externo. Reinicie el kernel y ejecute todas las celdas en orden. Normalmente solo tendrá que revisar la celda **Configuración de rutas**.


In [ ]:
from pathlib import Path
import gc
import json
import os
import subprocess
import sys
import time
import warnings

import pandas as pd
from IPython.display import display

# Localiza de forma portátil la raíz del repositorio que contiene intergate/.
here = Path.cwd().resolve()
candidates = [here, *here.parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'intergate').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'No se encuentra la carpeta intergate/. Abra el notebook dentro '
        'del repositorio InterGATE.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Raíz del código:', PROJECT_ROOT)


## Configuración de rutas

Los valores por defecto corresponden a la estructura de este repositorio. Modifique únicamente las rutas que estén en otra ubicación. `FORCE_REBUILD_ALL=False` protege los resultados existentes y permite reanudar una ejecución completa.


In [ ]:
# ── Rutas portátiles del repositorio ───────────────────────────────────────
DATA_PREPARE_DIR = Path(
    os.environ.get('INTERGATE_DATA_PREPARE_DIR', PROJECT_ROOT / 'data_prepare')
).expanduser().resolve()
ARTIFACTS_ROOT = Path(
    os.environ.get(
        'INTERGATE_ARTIFACTS_ROOT',
        PROJECT_ROOT / 'artifacts' / 'intrinsic4',
    )
).expanduser().resolve()
CACHE_ROOT = Path(
    os.environ.get('INTERGATE_CACHE_ROOT', PROJECT_ROOT / 'cache')
).expanduser().resolve()
EXTERNAL_DATA_DIR = Path(
    os.environ.get(
        'INTERGATE_EXTERNAL_DATA_DIR', PROJECT_ROOT / 'data' / 'external'
    )
).expanduser().resolve()
RESULTS_ROOT = Path(
    os.environ.get('INTERGATE_RESULTS_ROOT', PROJECT_ROOT / 'results_v6')
).expanduser().resolve()

default_external = RESULTS_ROOT / '07_external_locked'
EXTERNAL_RESULTS_DIR = Path(
    os.environ.get('INTERGATE_EXISTING_EXTERNAL_RESULTS', default_external)
).expanduser().resolve()

BUNDLE_ROOT = ARTIFACTS_ROOT / 'INTERGATE_V6'
BENCH_ROOT = RESULTS_ROOT / '11_paired_predictive_benchmark'

# ── Opciones de cómputo ─────────────────────────────────────────────
INSTALL_DEPENDENCIES = False  # True solo si faltan paquetes.
XGBOOST_DEVICE = 'cuda'       # Cambiar a 'cpu' si XGBoost no usa CUDA.
N_JOBS = -1                   # Todos los hilos disponibles para RF/XGBoost.
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 20260822
FORCE_REBUILD_ALL = False     # True rehace deliberadamente las tres fases.

# Deben fijarse antes de importar intergate.config.CFG.
os.environ['INTERGATE_DATA_PREPARE_DIR'] = str(DATA_PREPARE_DIR)
os.environ['INTERGATE_ARTIFACTS_ROOT'] = str(ARTIFACTS_ROOT)
os.environ['INTERGATE_CACHE_ROOT'] = str(CACHE_ROOT)
os.environ['INTERGATE_EXTERNAL_DATA_DIR'] = str(EXTERNAL_DATA_DIR)

configuration = pd.Series({
    'code': PROJECT_ROOT,
    'data_prepare': DATA_PREPARE_DIR,
    'artifacts': ARTIFACTS_ROOT,
    'cache': CACHE_ROOT,
    'external_data': EXTERNAL_DATA_DIR,
    'existing_external_results': EXTERNAL_RESULTS_DIR,
    'benchmark_output': BENCH_ROOT,
    'xgboost_device': XGBOOST_DEVICE,
    'n_jobs': N_JOBS,
    'bootstrap': N_BOOTSTRAP,
    'force_rebuild_all': FORCE_REBUILD_ALL,
}, name='value')
display(configuration.to_frame())


In [ ]:
# Instalación opcional. No modifica el entorno si INSTALL_DEPENDENCIES=False.
if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)
    ])
    print('Dependencias instaladas. Reinicie el kernel y vuelva a ejecutar.')
else:
    print('Instalación omitida; se usará el entorno actual.')


## Preflight

La siguiente celda detiene la ejecución antes de entrenar si falta alguna matriz, predicción externa o bundle de una de las cinco semillas.


In [ ]:
SEEDS = (1234, 2025, 7, 99, 31337)

required_files = [
    DATA_PREPARE_DIR / 'output_mcombat' / 'expr_train_mcombat.csv',
    DATA_PREPARE_DIR / 'output_mcombat' / 'expr_test_mcombat.csv',
    DATA_PREPARE_DIR / 'output_mcombat' / 'meta_train.csv',
    DATA_PREPARE_DIR / 'output_mcombat' / 'meta_test.csv',
    EXTERNAL_RESULTS_DIR / 'external_predictions.csv.gz',
    EXTERNAL_RESULTS_DIR / 'external_evaluation_COMPLETE.json',
]
for seed in SEEDS:
    seed_dir = BUNDLE_ROOT / f'seed_{seed}'
    required_files.extend([
        seed_dir / 'meta.json',
        seed_dir / 'graph_bundle.pt',
        seed_dir / 'model_pruned.pt',
    ])

audit = pd.DataFrame({
    'path': [str(path) for path in required_files],
    'exists': [path.is_file() for path in required_files],
})
display(audit)
missing = audit.loc[~audit['exists'], 'path'].tolist()
if missing:
    raise FileNotFoundError(
        'Faltan archivos obligatorios. Corrija la celda de rutas:\n- '
        + '\n- '.join(missing)
    )
print(f'Preflight correcto: {len(required_files)} archivos localizados.')


In [ ]:
import joblib
import numpy as np
import sklearn
import torch
import xgboost

if 'intergate.config' in sys.modules:
    warnings.warn(
        'intergate.config ya estaba importado. Si cambió las rutas, '
        'reinicie el kernel antes de continuar.',
        RuntimeWarning,
    )

from intergate.config import CFG
from intergate.predictive_benchmark import (
    evaluate_external,
    fit_development,
    summarize_predictions,
)
from intergate.runtime_setup import (
    assert_transport_protocol,
    assert_v6_protocol,
    prepare_runtime,
)

assert_v6_protocol(CFG)
assert_transport_protocol(CFG)

active_xgboost_device = str(XGBOOST_DEVICE)
if active_xgboost_device.lower() != 'cpu' and not torch.cuda.is_available():
    warnings.warn('CUDA no está disponible; XGBoost se ejecutará en CPU.')
    active_xgboost_device = 'cpu'

versions = pd.Series({
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'xgboost': xgboost.__version__,
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'xgboost_device': active_xgboost_device,
}, name='version')
display(versions.to_frame())


## Fase 1 · Ajuste y congelación en desarrollo

Se ajustan 15 modelos: tres algoritmos por cinco semillas. Todos utilizan las mismas filas de entrenamiento, las mismas filas de validación y el mismo orden de 7.131 genes que InterGATE. Esta es la fase más lenta.


In [ ]:
development_manifest = BENCH_ROOT / 'development_benchmark_manifest.json'
development_predictions = BENCH_ROOT / 'development_predictions.csv.gz'

development_complete = (
    development_manifest.is_file() and development_predictions.is_file()
)
if development_complete and not FORCE_REBUILD_ALL:
    manifest = json.loads(development_manifest.read_text(encoding='utf-8'))
    if manifest.get('status') != 'development_models_frozen':
        raise RuntimeError('El manifiesto de desarrollo existe pero no está completo.')
    print('Fase 1 ya completa; no se recalcula:', development_manifest)
else:
    started = time.perf_counter()
    runtime_development = prepare_runtime(
        CFG,
        include_external=False,
        build_biology_evidence=False,
    )
    development_outputs = fit_development(
        runtime_development,
        intergate_bundle_root=BUNDLE_ROOT,
        output_root=BENCH_ROOT,
        xgboost_device=active_xgboost_device,
        n_jobs=N_JOBS,
        external_complete_marker=(
            EXTERNAL_RESULTS_DIR / 'external_evaluation_COMPLETE.json'
        ),
        allow_posthoc=True,
        force=FORCE_REBUILD_ALL,
    )
    elapsed = (time.perf_counter() - started) / 60.0
    print(f'Fase 1 completada en {elapsed:.1f} min')
    display(pd.Series(development_outputs, name='path').to_frame())
    del runtime_development
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Fase 2 · Evaluación externa sin reajuste

Se cargan los objetos tabulares congelados y se aplican a las mismas muestras externas ya utilizadas por InterGATE. Las predicciones externas existentes de InterGATE se incorporan sin volver a ejecutar la red.


In [ ]:
external_root = BENCH_ROOT / 'external'
external_predictions = external_root / 'external_predictions_paired.csv.gz'
external_completion = external_root / 'external_benchmark_COMPLETE.json'

external_complete = external_predictions.is_file() and external_completion.is_file()
if external_complete and not FORCE_REBUILD_ALL:
    completion = json.loads(external_completion.read_text(encoding='utf-8'))
    if completion.get('status') != 'complete':
        raise RuntimeError('La evaluación externa existe pero no está completa.')
    print('Fase 2 ya completa; no se recalcula:', external_completion)
else:
    started = time.perf_counter()
    runtime_external = prepare_runtime(
        CFG,
        include_external=True,
        build_biology_evidence=False,
    )
    external_outputs = evaluate_external(
        runtime_external,
        development_output_root=BENCH_ROOT,
        output_root=external_root,
        existing_intergate_predictions=(
            EXTERNAL_RESULTS_DIR / 'external_predictions.csv.gz'
        ),
        force=FORCE_REBUILD_ALL,
    )
    elapsed = (time.perf_counter() - started) / 60.0
    print(f'Fase 2 completada en {elapsed:.1f} min')
    display(pd.Series(external_outputs, name='path').to_frame())
    del runtime_external
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Fase 3 · Métricas, bootstrap y S12

El bootstrap utiliza 2.000 réplicas estratificadas por cohorte × clase real. Las diferencias pareadas se calculan como InterGATE menos cada comparador sobre las mismas observaciones remuestreadas.


In [ ]:
summary_root = BENCH_ROOT / 'summary'
summary_manifest = summary_root / 'predictive_benchmark_manifest.json'
s12_overall_path = summary_root / 'supplementary_table_s12_overall.csv'
s12_class_path = summary_root / 'supplementary_table_s12_by_class.csv'

summary_complete = (
    summary_manifest.is_file()
    and s12_overall_path.is_file()
    and s12_class_path.is_file()
)
if summary_complete and not FORCE_REBUILD_ALL:
    summary_payload = json.loads(summary_manifest.read_text(encoding='utf-8'))
    if summary_payload.get('status') != 'complete':
        raise RuntimeError('El resumen existe pero no está completo.')
    print('Fase 3 ya completa; no se recalcula:', summary_manifest)
else:
    started = time.perf_counter()
    summary_outputs = summarize_predictions(
        [development_predictions, external_predictions],
        output_dir=summary_root,
        n_bootstrap=N_BOOTSTRAP,
        random_seed=BOOTSTRAP_SEED,
    )
    elapsed = (time.perf_counter() - started) / 60.0
    print(f'Fase 3 completada en {elapsed:.1f} min')
    display(pd.Series(summary_outputs, name='path').to_frame())


## Resultado externo principal

La tabla mostrada combina la media ± DE entre las cinco semillas con la estimación del ensamblado y su IC del 95 % bootstrap.


In [ ]:
s12_overall = pd.read_csv(s12_overall_path)
s12_by_class = pd.read_csv(s12_class_path)
paired_differences = pd.read_csv(
    summary_root / 'paired_model_differences_95ci.csv'
)

external_pooled = s12_overall.loc[
    s12_overall['split'].eq('locked_external')
    & s12_overall['scope'].eq('pooled'),
    [
        'model', 'metric', 'mean', 'sd', 'estimate',
        'ci_95_low', 'ci_95_high', 'n_samples', 'n_seeds',
    ],
].sort_values(['metric', 'model'])
display(external_pooled.reset_index(drop=True))

print('\nArchivos que debe devolver para actualizar el manuscrito:')
for path in [
    s12_overall_path,
    s12_class_path,
    summary_root / 'paired_model_differences_95ci.csv',
    summary_root / 'confusion_mean_across_seeds.csv',
    summary_root / 'bootstrap_luma_lumb_exchange_95ci.csv',
    summary_manifest,
]:
    print(' -', path.resolve())

print('\nBenchmark completo. Comprima y suba la carpeta:')
print(BENCH_ROOT.resolve())


## Referencias metodológicas

- Brouard C, et al. *Briefings in Bioinformatics*. 2024;25:bbae027. https://doi.org/10.1093/bib/bbae027
- Efron B, Tibshirani RJ. *An Introduction to the Bootstrap*. Chapman & Hall/CRC; 1993. https://doi.org/10.1201/9780429246593
